# Phase 3 — Machine Learning Modelling

**Project:** “Machine Learning approaches for accelerating antibiotic susceptibility testing (AST) in microfluidic systems” 

**Date:** May 2026 

## 1. Introduction

This notebook trains and evaluates multiple machine learning models to classify bacterial growth based on the engineered features from Phase 9. 

Two experimental setups are compared: one including concentration as a feature, and one without it, to assess the impact of data leakage on model performance.

### Objectives
- Load and prepare the labelled feature matrix for modelling
- Split the dataset by experiment into train and test sets
- Train and evaluate Logistic Regression and Random Forest models with all features
- Retrain and compare all models without the concentration feature
- Identify the best performing model across both setups

### Input
- `ecoli_data_ml.csv` — labelled feature matrix from Phase 9

### Models Evaluated
    | Logistic Regression | Random Forest | Support Vector Machine | K-Nearest Neighbours | Naive Bayes |

## Table of Contents
1. [Introduction](#1-introduction)
2. [Imports & Setup](#2-imports--setup)
3. [Data Loading & Preparation](#3-data-loading--preparation)
4. [Concentration](#4-concetration)
5. [Train / Test Split](#5-train--test-split)
6. [Logistic Regression](#6-logistic-regression)
7. [Random Forest](#7-random-forest)
8. [Model — Without Concentration Feature](#8-model---without-concentration-feature)
9. [Logistic Regression](#9-logistic-regression)
10. [Random Forest](#10-random-forest)
11. [Support Vector Machine](#11-support-vector-machine)
12. [K-Nearest Neighbours](#12-k-nearest-neighbours)
13. [Naive Bayes](#13-naive-bayes)
14. [Conclusions](#14-conclusion)

## 2. Imports & Setup

In [1]:
# -- Data manipulation ------------------------------------------
import numpy as np
import pandas as pd

# -- Machine learning -------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

## 3. Data Loading & Preparation

### 3.1 Load dataset

In [2]:
# Load the labelled feature matrix from Phase 2
for_ml = pd.read_csv("ecoli_data_ml.csv")

print(f"Dataset shape: {for_ml.shape}")
for_ml.head()

Dataset shape: (45342, 14)


,experiencia,tubo_id,antibiotico,concentração,tempo,gvr,std_inicial,std_atual,std_tubo,slope,delta,AUC,Derivada,label
0,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0.0,0.000000,0.069883,0.008784,0.008784,0.000000,0.000000,0.000000,0.000000,0.000000,1
1,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0.0,0.166667,0.069883,0.008784,0.008784,0.000000,0.000000,0.000000,0.011647,0.000000,1
2,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0.0,0.333333,-0.139765,0.008784,0.017568,0.098829,-0.628944,-0.209648,0.005824,-1.257887,1
3,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0.0,0.500000,-0.199132,0.008784,0.025127,0.121492,-0.538028,-0.269014,-0.022418,-0.356198,1
4,ampicillin_20230508_5e6,ampicillin_20230508_5e6_0,ampicillin,0.0,0.666667,-0.119479,0.008784,0.024350,0.112185,-0.284042,-0.189362,-0.048969,0.477916,1


## 4. Concetration

### 4.1 Remove outlier and irrelevant columns

In [3]:
# Remove outlier experiment identified in Phase 3
dataset = for_ml[for_ml["experiencia"] != "ampicillin_20230518_5e5"]

# Drop non-informative columns
dataset = dataset.drop(columns=["antibiotico"])

print(f"Dataset shape after cleaning: {dataset.shape}")

Dataset shape after cleaning: (44352, 13)


### 4.2 Define features and label

In [4]:
colunas = list(dataset.head(0))
features = colunas[2: -1]
label = colunas[-1]

print("Features:", features)

Features: ['concentração', 'tempo', 'gvr', 'std_inicial', 'std_atual', 'std_tubo', 'slope', 'delta', 'AUC', 'Derivada']


## 5. Train / Test Split

The dataset is split by experiment to simulate real-world generalisation:

    the model is trained on all experiments except the last two, which are held out as independent test sets.

In [5]:
experiencias = dataset["experiencia"].unique()

exp_teste1 = "ampicillin_20230518_5e4"
exp_teste2 = "ampicillin_20230518_5e6"

# train
df_train = dataset[~dataset["experiencia"].isin([exp_teste1, exp_teste2])]

# test 1 and test 2
df_test1 = dataset[dataset["experiencia"] == exp_teste1]
df_test2 = dataset[dataset["experiencia"] == exp_teste2]

X_train = df_train[features]
y_train = df_train["label"]

X_test1 = df_test1[features]
y_test1 = df_test1["label"]

X_test2 = df_test2[features]
y_test2 = df_test2["label"]

print(f"Train size : {len(df_train)} rows")
print(f"Test 1 size: {len(df_test1)} rows  ({exp_teste1})")
print(f"Test 2 size: {len(df_test2)} rows  ({exp_teste2})")

Train size : 42372 rows
Test 1 size: 990 rows  (ampicillin_20230518_5e4)
Test 2 size: 990 rows  (ampicillin_20230518_5e6)


In [6]:
print(f"Testeset 1 shape after: {df_test1.shape}")
print(f"Testeset 1 shape after: {df_test2.shape}")
print(f"Train set shape after: {X_train.shape}")

Testeset 1 shape after: (990, 13)
Testeset 1 shape after: (990, 13)
Train set shape after: (42372, 10)


## 6. Logistic Regression

### 6.1 Train

In [7]:
# Criar modelo
modelo_logistic_noconc = LogisticRegression(max_iter=5000)

# Treinar
modelo_logistic_noconc.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

### 6.2 Evaluate on Test 1

In [8]:
# Fazer previsões
y_pred_lr_noconc = modelo_logistic_noconc.predict(X_test1)
y_prob_lr_noconc = modelo_logistic_noconc.predict_proba(X_test1)[:,1]

print(" Logistic Regression - Test 1 ")
print(f"Accuracy : {accuracy_score(y_test1, y_pred_lr_noconc)}")
print(f"ROC-AUC  : {roc_auc_score(y_test1, y_prob_lr_noconc)}")
print()
print(classification_report(y_test1, y_pred_lr_noconc))
print("Confusion Matrix (TN  FP / FN  TP):")
print(confusion_matrix(y_test1, y_pred_lr_noconc))

 Logistic Regression - Test 1 
Accuracy : 0.8939393939393939
ROC-AUC  : 0.964032241607999

              precision    recall  f1-score   support

           0       0.90      0.89      0.89       495
           1       0.89      0.90      0.89       495

    accuracy                           0.89       990
   macro avg       0.89      0.89      0.89       990
weighted avg       0.89      0.89      0.89       990

Confusion Matrix (TN  FP / FN  TP):
[[439  56]
 [ 49 446]]


### 6.3 Cross-validation

In [9]:
cv_scores_lr = cross_val_score(modelo_logistic_noconc, X_train, y_train, cv=5, scoring="roc_auc")

print("Cross-validation ROC-AUC scores:", cv_scores_lr)
print(f"Mean: {cv_scores_lr.mean()}")

Cross-validation ROC-AUC scores: [0.96104608 0.98526385 0.98922709 0.91323038 0.70658916]
Mean: 0.91107131467797


## 7. Random Forest

### 7.1 Train

In [10]:
# Criar modelo
modelo_rf_conc = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Treinar
modelo_rf_conc.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

### 7.2 Evaluate on Test 1

In [11]:
y_prob_rf_conc = modelo_rf_conc.predict_proba(X_test1)[:, 1]
y_pred_rf_conc    = (y_prob_rf_conc > 0.8).astype(int)   # custom threshold

print(" Random Forest - Test 1")
print(f"Accuracy : {accuracy_score(y_test1, y_pred_rf_conc):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test1, y_prob_rf_conc):.4f}")
print()
print(classification_report(y_test1, y_pred_rf_conc))
print("Confusion Matrix (TN  FP / FN  TP):")
print(confusion_matrix(y_test1, y_pred_rf_conc))

 Random Forest - Test 1
Accuracy : 0.7919
ROC-AUC  : 0.9389

              precision    recall  f1-score   support

           0       0.71      1.00      0.83       495
           1       1.00      0.58      0.74       495

    accuracy                           0.79       990
   macro avg       0.85      0.79      0.78       990
weighted avg       0.85      0.79      0.78       990

Confusion Matrix (TN  FP / FN  TP):
[[495   0]
 [206 289]]


### 7.3 Cross-validation

In [12]:
cv_scores_rf = cross_val_score(modelo_rf_conc, X_train, y_train, cv=5, scoring="roc_auc")

print("Cross-validation ROC-AUC scores:", cv_scores_rf)
print(f"Mean: {cv_scores_rf.mean():.4f}")

Cross-validation ROC-AUC scores: [0.93545678 0.96399282 0.97200055 0.94415948 0.97495053]
Mean: 0.9581


### 7.4 Feature importance

In [13]:
importance = pd.Series(modelo_rf_conc.feature_importances_, index=X_train.columns)

print("Top 20 most important features:")
print(importance.sort_values(ascending=False).head(20))

Top 20 most important features:
gvr             0.243838
AUC             0.179417
concentração    0.151240
delta           0.132279
slope           0.099522
std_inicial     0.055874
std_atual       0.055300
std_tubo        0.046590
tempo           0.027959
Derivada        0.007981
dtype: float64


## 8. Model - Without Concentration Feature

Since concentration was likely causing data leakage (the model was memorising the feature rather than learning biological patterns), all models were retrain after removing it from the feature set.

### 8.1 — Drop concentration and redefine features

In [14]:
# Remove outlier experiment identified during EDA
dataset = for_ml[for_ml["experiencia"] != "20230518_5e5"]
# Drop string columns
df_no_conc = dataset.drop(columns=["concentração"])
df_no_conc = df_no_conc.drop(columns=["antibiotico"])

columns          = list(df_no_conc.head(0))
features_noconc = columns[2:-1]
label = columns[-1]

print(f"Dataset shape after cleaning: {df_no_conc.shape}")

Dataset shape after cleaning: (45342, 12)


### 8.2 — Train / test split

In [15]:
X_train_nc = df_train.drop(columns=["concentração"])[features_noconc]
X_test1_nc = df_test1.drop(columns=["concentração"])[features_noconc]
X_test2_nc = df_test2.drop(columns=["concentração"])[features_noconc]

In [16]:
print(f"Testeset 1 shape after: {X_test1_nc.shape}")
print(f"Testeset 1 shape after: {X_test2_nc.shape}")
print(f"Train set shape after: {X_train_nc.shape}")

Testeset 1 shape after: (990, 9)
Testeset 1 shape after: (990, 9)
Train set shape after: (42372, 9)


## 9. Logistic Regression

### 9.1 Train and evaluate

In [17]:
model_lr = LogisticRegression(max_iter=5000)
model_lr.fit(X_train_nc, y_train)

y_pred_lr       = model_lr.predict(X_test1_nc)
y_prob_lr = model_lr.predict_proba(X_test1_nc)[:, 1]

print("Logistic Regression (no concentration) — Test 1")
print(f"Accuracy : {accuracy_score(y_test1, y_pred_lr)}")
print(f"ROC-AUC  : {roc_auc_score(y_test1, y_prob_lr)}")
print()
print(classification_report(y_test1, y_pred_lr))
print("Confusion Matrix (TN  FP / FN  TP):")
print(confusion_matrix(y_test1, y_pred_lr))

Logistic Regression (no concentration) — Test 1
Accuracy : 0.7393939393939394
ROC-AUC  : 0.8619569431690643

              precision    recall  f1-score   support

           0       0.77      0.68      0.72       495
           1       0.71      0.80      0.75       495

    accuracy                           0.74       990
   macro avg       0.74      0.74      0.74       990
weighted avg       0.74      0.74      0.74       990

Confusion Matrix (TN  FP / FN  TP):
[[337 158]
 [100 395]]


### 9.2 Cross-validation

In [18]:
cv_lr_nc = cross_val_score(model_lr, X_train_nc, y_train, cv=5, scoring="roc_auc")

print("Cross-validation ROC-AUC:", cv_lr_nc)
print(f"Mean: {cv_lr_nc.mean()}")

Cross-validation ROC-AUC: [0.89883241 0.98566031 0.98545837 0.90864234 0.75399019]
Mean: 0.9065167238784602


## 10. Random Forest

### 10.1 Train and evaluate

In [19]:
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train_nc, y_train)

y_prob_rf = model_rf.predict_proba(X_test1_nc)[:, 1]
y_pred_rf       = (y_prob_rf > 0.8).astype(int)

print("Random Forest (no concentration) — Test 1")
print(f"Accuracy : {accuracy_score(y_test1, y_pred_rf)}")
print(f"ROC-AUC  : {roc_auc_score(y_test1, y_prob_rf)}")
print()
print(classification_report(y_test1, y_pred_rf))
print("Confusion Matrix (TN  FP / FN  TP):")
print(confusion_matrix(y_test1, y_pred_rf))

Random Forest (no concentration) — Test 1
Accuracy : 0.7525252525252525
ROC-AUC  : 0.8833568003264973

              precision    recall  f1-score   support

           0       0.68      0.96      0.79       495
           1       0.93      0.55      0.69       495

    accuracy                           0.75       990
   macro avg       0.80      0.75      0.74       990
weighted avg       0.80      0.75      0.74       990

Confusion Matrix (TN  FP / FN  TP):
[[473  22]
 [223 272]]


### 10.2 Cross-validation

In [20]:
cv_rf_nc = cross_val_score(model_rf, X_train_nc, y_train, cv=5, scoring="roc_auc")

print("Cross-validation ROC-AUC:", cv_rf_nc)
print(f"Mean: {cv_rf_nc.mean():.4f}")

Cross-validation ROC-AUC: [0.93937566 0.97388367 0.96098519 0.90103801 0.946209  ]
Mean: 0.9443


### 10.3 Feature importance

In [21]:
importance = pd.Series(model_rf.feature_importances_, index=X_train_nc.columns)

print("Top 20 most important features:")
print(importance.sort_values(ascending=False).head(20))

Top 20 most important features:
gvr            0.352531
AUC            0.220435
delta          0.113426
std_atual      0.069169
slope          0.066878
std_tubo       0.062334
std_inicial    0.050335
tempo          0.043121
Derivada       0.021770
dtype: float64


## 11. Support Vector Machine

### 11.1 Scale & train

In [22]:
# SVM and KNN require scaled features
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_nc)
X_test1_scaled = scaler.transform(X_test1_nc)

### 11.2 Evaluate

In [23]:
model_svc = SVC(kernel="rbf", probability=True)
model_svc.fit(X_train_scaled, y_train)

y_pred_svc = model_svc.predict(X_test1_scaled)
y_prob_svc = model_svc.predict_proba(X_test1_scaled)[:, 1]

print(" SVM (no concentration) — Test 1")
print(f"Accuracy : {accuracy_score(y_test1, y_pred_svc)}")
print(f"ROC-AUC  : {roc_auc_score(y_test1, y_prob_svc)}")
print()
print(classification_report(y_test1, y_pred_svc))
print("Confusion Matrix (TN  FP / FN  TP):")
print(confusion_matrix(y_test1, y_pred_svc))

 SVM (no concentration) — Test 1
Accuracy : 0.7585858585858586
ROC-AUC  : 0.8390898887868585

              precision    recall  f1-score   support

           0       0.81      0.67      0.74       495
           1       0.72      0.85      0.78       495

    accuracy                           0.76       990
   macro avg       0.77      0.76      0.76       990
weighted avg       0.77      0.76      0.76       990

Confusion Matrix (TN  FP / FN  TP):
[[332 163]
 [ 76 419]]


In [24]:
cv_scv_nc = cross_val_score(model_svc, X_train_nc, y_train, cv=5, scoring="roc_auc")

print("Cross-validation ROC-AUC:", cv_scv_nc)
print(f"Mean: {cv_scv_nc.mean():.4f}")

Cross-validation ROC-AUC: [0.91668947 0.98365231 0.98126115 0.89992522 0.91016955]
Mean: 0.9383


## 12. K-Nearest Neighbours

### 12.1 Train & evaluate

In [25]:
model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train_scaled, y_train)

y_pred_knn       = model_knn.predict(X_test1_scaled)
y_prob_knn   = model_knn.predict_proba(X_test1_scaled)[:, 1]

print(" KNN (no concentration) — Test 1 ")
print(f"Accuracy : {accuracy_score(y_test1, y_pred_knn)}")
print(f"ROC-AUC  : {roc_auc_score(y_test1, y_prob_knn)}")
print()
print(classification_report(y_test1, y_pred_knn))
print("Confusion Matrix (TN  FP / FN  TP):")
print(confusion_matrix(y_test1, y_pred_knn))

 KNN (no concentration) — Test 1 
Accuracy : 0.7333333333333333
ROC-AUC  : 0.8267707376798286

              precision    recall  f1-score   support

           0       0.74      0.73      0.73       495
           1       0.73      0.74      0.74       495

    accuracy                           0.73       990
   macro avg       0.73      0.73      0.73       990
weighted avg       0.73      0.73      0.73       990

Confusion Matrix (TN  FP / FN  TP):
[[359 136]
 [128 367]]


In [26]:
cv_knn_nc = cross_val_score(model_knn, X_train_nc, y_train, cv=5, scoring="roc_auc")

print("Cross-validation ROC-AUC:", cv_knn_nc)
print(f"Mean: {cv_knn_nc.mean():.4f}")

Cross-validation ROC-AUC: [0.90998587 0.96262126 0.9520333  0.84824076 0.88827398]
Mean: 0.9122


### 13. Naive Bayes

In [27]:
model_nb   = GaussianNB()
model_nb.fit(X_train_nc, y_train)

y_pred_nb     = model_nb.predict(X_test1_nc)
y_prob_nb  = model_nb.predict_proba(X_test1_nc)[:, 1]

print(" Naive Bayes (no concentration) — Test 1 ")
print(f"Accuracy : {accuracy_score(y_test1, y_pred_nb)}")
print(f"ROC-AUC  : {roc_auc_score(y_test1, y_prob_nb)}")
print()
print(classification_report(y_test1, y_pred_nb))
print("Confusion Matrix (TN  FP / FN  TP):")
print(confusion_matrix(y_test1, y_pred_nb))

 Naive Bayes (no concentration) — Test 1 
Accuracy : 0.7434343434343434
ROC-AUC  : 0.8497132945617794

              precision    recall  f1-score   support

           0       0.66      0.99      0.79       495
           1       0.98      0.49      0.66       495

    accuracy                           0.74       990
   macro avg       0.82      0.74      0.73       990
weighted avg       0.82      0.74      0.73       990

Confusion Matrix (TN  FP / FN  TP):
[[491   4]
 [250 245]]


In [28]:
cv_nb_nc = cross_val_score(model_nb, X_train_nc, y_train, cv=5, scoring="roc_auc")

print("Cross-validation ROC-AUC:", cv_nb_nc)
print(f"Mean: {cv_nb_nc.mean():.4f}")

Cross-validation ROC-AUC: [0.86031802 0.96866414 0.92316285 0.86339533 0.64653738]
Mean: 0.8524


## 14. Conclusion

This notebook trained and evaluated five machine learning models for bacterial growth classification using time-series features derived from GVR measurements.

In [29]:
# Model comparison summary — without concentration feature
results = {
    "Logistic Regression": {
        "accuracy": accuracy_score(y_test1, y_pred_lr),
        "roc_auc" : roc_auc_score(y_test1, y_prob_lr),
        "cv_mean" : cross_val_score(model_lr, X_train_nc, y_train, cv=5, scoring="roc_auc").mean()
    },
    "Random Forest": {
        "accuracy": accuracy_score(y_test1, y_pred_rf),
        "roc_auc" : roc_auc_score(y_test1, y_prob_rf),
        "cv_mean" : cross_val_score(model_rf, X_train_nc, y_train, cv=5, scoring="roc_auc").mean()
    },
    "Support Vector Machine": {
        "accuracy": accuracy_score(y_test1, y_pred_svc),
        "roc_auc" : roc_auc_score(y_test1, y_prob_svc), 
        "cv_mean" : cross_val_score(model_svc, X_train_scaled, y_train, cv=5, scoring="roc_auc").mean()
    },
    "K-Nearest Neighbours": {
        "accuracy": accuracy_score(y_test1, y_pred_knn),
        "roc_auc" : roc_auc_score(y_test1, y_prob_knn),
        "cv_mean" : cross_val_score(model_knn, X_train_scaled, y_train, cv=5, scoring="roc_auc").mean()
    },
    "Naive Bayes": {
        "accuracy": accuracy_score(y_test1, y_pred_nb),
        "roc_auc" : roc_auc_score(y_test1, y_prob_nb),
        "cv_mean" : cross_val_score(model_nb, X_train_nc, y_train, cv=5, scoring="roc_auc").mean()
    }
}

# Display as DataFrame
df_results = pd.DataFrame(results).T
df_results.columns = ["Accuracy", "ROC-AUC", "Cross Value Score (mean)"]
df_results = df_results.round(4)

display(df_results)

,Accuracy,ROC-AUC,Cross Value Score (mean)
Logistic Regression,0.7394,0.8620,0.9065
Random Forest,0.7525,0.8834,0.9443
Support Vector Machine,0.7586,0.8391,0.9493
K-Nearest Neighbours,0.7333,0.8268,0.9042
Naive Bayes,0.7434,0.8497,0.8524
